In [1]:

import os
import re
import sys

import numpy as np
import pandas as pd

IN_PATH = "online_retail_II.csv"
OUT = "day_1_outputs"
MIN_N = 50          # minimum customers for a product to count in the gateway analysis
WIN_REPEAT = 180    # "repeat" = second order within this many days of the first
WIN_LTV = 365       # LTV = revenue within this many days of the first order

os.makedirs(OUT, exist_ok=True)
lines = []


def say(text=""):
    text = str(text)
    lines.append(text)
    try:
        print(text)
    except UnicodeEncodeError:
        print(text.encode("ascii", "replace").decode())


def show(df):
    say(df.to_string(index=False))



# 1. Load

CANON = {
    "invoice": "invoice", "invoiceno": "invoice",
    "stockcode": "stock",
    "description": "desc",
    "quantity": "qty",
    "invoicedate": "date",
    "price": "price", "unitprice": "price",
    "customerid": "cust",
    "country": "country",
}

df = None
for enc in ("utf-8-sig", "latin-1"):
    try:
        head = pd.read_csv(IN_PATH, nrows=0, encoding=enc)
        colmap = {c: CANON.get(re.sub("[^a-z]", "", c.lower())) for c in head.columns}
        colmap = {c: n for c, n in colmap.items() if n}
        text_cols = {c: str for c, n in colmap.items() if n in ("invoice", "stock", "desc", "date")}
        df = pd.read_csv(IN_PATH, encoding=enc, usecols=list(colmap), dtype=text_cols)
        df = df.rename(columns=colmap)
        break
    except UnicodeDecodeError:
        continue
    except FileNotFoundError:
        sys.exit(f"Cannot find '{IN_PATH}'. Put the CSV in this folder or pass its name.")

if df is None:
    sys.exit("Could not read the file.")
missing = {"invoice", "stock", "qty", "date", "price", "cust"} - set(df.columns)
if missing:
    sys.exit(f"Missing expected columns: {missing}. Columns found: {list(df.columns)}")
if "desc" not in df:
    df["desc"] = ""
if "country" not in df:
    df["country"] = ""

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["qty"] = pd.to_numeric(df["qty"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["cust"] = pd.to_numeric(df["cust"], errors="coerce").astype("Int64")
df["invoice"] = df["invoice"].astype(str).str.strip().str.upper()
df["stock"] = df["stock"].astype(str).str.strip().str.upper()
df["rev"] = df["qty"] * df["price"]


# 2. Clean (every step logged)

log = []


def step(name, mask, note=""):
    """Log and apply removal of rows where mask is True. Returns kept frame."""
    global df
    removed = df[mask]
    log.append({
        "step": name,
        "rows_removed": int(mask.sum()),
        "value_removed": round(float(removed["rev"].sum()), 2),
        "rows_remaining": int((~mask).sum()),
        "note": note,
    })
    kept = df[~mask]
    return removed, kept


log.append({"step": "Rows loaded", "rows_removed": 0, "value_removed": 0.0,
            "rows_remaining": len(df), "note": ""})
n_dup = int(df.duplicated().sum())

_, df = step("Unparseable date, quantity or price",
             df["date"].isna() | df["qty"].isna() | df["price"].isna())

data_end = df["date"].max()
data_start = df["date"].min()

returns, df = step("Cancellations/returns (invoice starts with C), set aside",
                   df["invoice"].str.startswith("C"),
                   "Value is negative. Kept separately, not netted off sales.")
_, df = step("Zero or negative quantity or price (not cancellations)",
             (df["qty"] <= 0) | (df["price"] <= 0), "Stock adjustments and bad-debt entries.")

base_rev = float(df["rev"].sum())  # all positive sales lines

nonprod, df = step("Non-product stock codes (postage, fees, manual, etc.)",
                   ~df["stock"].str.match(r"^\d{5}"),
                   "Kept only codes starting with 5 digits.")
nocust, df = step("Missing Customer ID", df["cust"].isna(),
                  "Cannot be tied to a customer, so excluded from customer analysis.")

# Analysis window: drop a partial final month
month_end = (data_end + pd.offsets.MonthEnd(0)).normalize()
last_full = data_end.to_period("M") if data_end.normalize() >= month_end else data_end.to_period("M") - 1
window_end = last_full.end_time.normalize()
last_idx = last_full.year * 12 + last_full.month
late, df = step(f"Dated after {window_end.date()} (partial final month)", df["date"] > window_end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1),
                "Partial months distort retention, so the window ends at the last full month.")

sales = df.copy()
analysis_rev = float(sales["rev"].sum())
log_df = pd.DataFrame(log)
log_df.to_csv(f"{OUT}/cleaning_log.csv", index=False)


# 3. Customer table

sales["order_day"] = sales["date"].dt.normalize()
sales["m_idx"] = sales["date"].dt.year * 12 + sales["date"].dt.month

orders = (sales.groupby(["cust", "order_day"], as_index=False)
          .agg(rev=("rev", "sum"), invoices=("invoice", "nunique"))
          .sort_values(["cust", "order_day"]))
orders["order_no"] = orders.groupby("cust").cumcount() + 1
first = orders[orders["order_no"] == 1].set_index("cust")
second = orders[orders["order_no"] == 2].set_index("cust")

cust = orders.groupby("cust").agg(first_order_day=("order_day", "min"),
                                  last_order_day=("order_day", "max"),
                                  order_days=("order_day", "nunique"),
                                  revenue=("rev", "sum"))
cust["first_order_value"] = first["rev"]
cust["second_order_day"] = second["order_day"]
cust["days_to_second"] = (cust["second_order_day"] - cust["first_order_day"]).dt.days
cust["cohort_idx"] = cust["first_order_day"].dt.year * 12 + cust["first_order_day"].dt.month
cust["cohort_month"] = cust["first_order_day"].dt.strftime("%Y-%m")
cust["repeat_ever"] = cust["order_days"] >= 2
cust["repeat_180d"] = cust["days_to_second"].le(WIN_REPEAT).astype(bool)
cust["eligible_180d"] = (cust["first_order_day"] + pd.Timedelta(days=WIN_REPEAT)) <= window_end
cust["eligible_365d"] = (cust["first_order_day"] + pd.Timedelta(days=WIN_LTV)) <= window_end

# The file starts in Dec 2009, so customers first seen that month include people who
# had already bought before the file began. Their true first order is unknown, so they
# are excluded from cohort, first-order, repeat and LTV analysis (but kept in RFM).
first_cohort = cust["cohort_idx"].min()
cust["is_new"] = cust["cohort_idx"] > first_cohort
cust["eligible_180d"] = cust["eligible_180d"] & cust["is_new"]
cust["eligible_365d"] = cust["eligible_365d"] & cust["is_new"]
cn = cust[cust["is_new"]].copy()

# LTV within 365 days of first order
s2 = sales[["cust", "order_day", "rev"]].copy()
s2["first_day"] = s2["cust"].map(cust["first_order_day"])
s2 = s2[(s2["order_day"] - s2["first_day"]).dt.days < WIN_LTV]
cust["ltv_365"] = s2.groupby("cust")["rev"].sum()


# 4. Cohort retention and LTV curves (calendar months)

def idx_label(i):
    return f"{(i - 1) // 12}-{(i - 1) % 12 + 1:02d}"


sales["cohort_idx"] = sales["cust"].map(cust["cohort_idx"])
sales["k"] = sales["m_idx"] - sales["cohort_idx"]
sn = sales[sales["cust"].isin(cn.index)]
act = (sn.drop_duplicates(["cust", "k"]).groupby(["cohort_idx", "k"]).size()
       .rename("active").reset_index())
rev_k = sn.groupby(["cohort_idx", "k"])["rev"].sum().rename("rev").reset_index()
sizes = cn.groupby("cohort_idx").size().rename("cohort_size").reset_index()
grid = pd.DataFrame([(c, k) for c in sizes["cohort_idx"] for k in range(0, last_idx - c + 1)],
                    columns=["cohort_idx", "k"])
cr = (grid.merge(act, how="left").merge(rev_k, how="left").merge(sizes, on="cohort_idx")
      .fillna({"active": 0, "rev": 0}))
cr["retention_rate"] = cr["active"] / cr["cohort_size"]
cr["cum_rev"] = cr.groupby("cohort_idx")["rev"].cumsum()
cr["ltv_per_customer"] = cr["cum_rev"] / cr["cohort_size"]
cr["cohort_month"] = cr["cohort_idx"].map(idx_label)
cr = cr.rename(columns={"k": "months_since_first", "active": "customers_active"})

cr[["cohort_month", "months_since_first", "cohort_size", "customers_active", "retention_rate"]] \
    .round(4).to_csv(f"{OUT}/cohort_retention.csv", index=False)
cr[["cohort_month", "months_since_first", "cohort_size", "cum_rev", "ltv_per_customer"]] \
    .round(2).to_csv(f"{OUT}/ltv_by_cohort.csv", index=False)


# 5. First-order value bands

_band, edges = pd.qcut(cn["first_order_value"], 4,
                       labels=["Q1 smallest", "Q2", "Q3", "Q4 largest"],
                       retbins=True)
cust["first_order_band"] = _band
rows = []
for b, g in cust.groupby("first_order_band", observed=True):
    e180 = g[g["eligible_180d"]]
    e365 = g[g["eligible_365d"]]
    rows.append({
        "band": b,
        "first_order_range": f"{edges[list(cust['first_order_band'].cat.categories).index(b)]:.0f} to "
                             f"{edges[list(cust['first_order_band'].cat.categories).index(b) + 1]:.0f}",
        "customers_all": len(g),
        "customers_eligible_180d": len(e180),
        "repeat_rate_180d": round(e180["repeat_180d"].mean(), 4) if len(e180) else np.nan,
        "customers_eligible_365d": len(e365),
        "avg_ltv_365": round(e365["ltv_365"].mean(), 2) if len(e365) else np.nan,
        "median_ltv_365": round(e365["ltv_365"].median(), 2) if len(e365) else np.nan,
    })
bands = pd.DataFrame(rows)
bands.to_csv(f"{OUT}/first_order_bands.csv", index=False)


# 6. Gateway products (what was in the first order, and who came back)

sales["first_day"] = sales["cust"].map(cust["first_order_day"])
fl = (sales[sales["order_day"] == sales["first_day"]][["cust", "stock"]]
      .drop_duplicates())
fl = fl.merge(cust[["eligible_180d", "repeat_180d"]], left_on="cust", right_index=True)
fl = fl[fl["eligible_180d"]]
baseline = float(cust[cust["eligible_180d"]]["repeat_180d"].mean())

gw = fl.groupby("stock").agg(customers=("cust", "nunique"),
                             repeaters=("repeat_180d", "sum")).reset_index()
gw["repeat_rate_180d"] = gw["repeaters"] / gw["customers"]
gw["lift_vs_baseline_pp"] = (gw["repeat_rate_180d"] - baseline) * 100
gw["meets_min_customers"] = gw["customers"] >= MIN_N
desc = (sales.dropna(subset=["desc"]).groupby("stock")["desc"]
        .agg(lambda s: s.value_counts().index[0]))
gw["description"] = gw["stock"].map(desc)
gw = gw.sort_values(["meets_min_customers", "repeat_rate_180d"], ascending=[False, False])
gw[["stock", "description", "customers", "repeaters", "repeat_rate_180d",
    "lift_vs_baseline_pp", "meets_min_customers"]].round(4).to_csv(
    f"{OUT}/gateway_products.csv", index=False)


# 7. RFM segments

snapshot = window_end + pd.Timedelta(days=1)
cust["recency_days"] = (snapshot - cust["last_order_day"]).dt.days
cust["r"] = pd.cut(cust["recency_days"], [-1, 30, 90, 180, 365, np.inf], labels=[5, 4, 3, 2, 1]).astype(int)
cust["f"] = pd.cut(cust["order_days"], [0, 1, 2, 4, 9, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
cust["m"] = pd.qcut(cust["revenue"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

conds = [
    (cust["r"] >= 4) & (cust["f"] >= 4),
    (cust["r"] >= 3) & (cust["f"] >= 3),
    (cust["r"] <= 2) & (cust["f"] >= 3),
    (cust["r"] >= 4) & (cust["f"] <= 2),
    (cust["r"] <= 2) & (cust["f"] <= 2),
]
names = ["Champions", "Loyal", "At risk", "New or recent", "Lost or low"]
cust["segment"] = np.select(conds, names, default="Needs attention")
actions = {
    "Champions": "Reward and protect: early access, loyalty perks, ask for referrals",
    "Loyal": "Keep engaged: cross-sell, bundles, reminders before usual reorder gap",
    "At risk": "Win back now: personal outreach and an incentive; they bought often but have gone quiet",
    "New or recent": "Secure the second order: follow-up within the usual repeat window",
    "Needs attention": "Nudge: reminders and relevant product suggestions",
    "Lost or low": "Low-cost reactivation only; do not over-invest",
}
seg = cust.groupby("segment").agg(customers=("revenue", "size"), revenue=("revenue", "sum"),
                                  avg_revenue=("revenue", "mean"),
                                  avg_order_days=("order_days", "mean"),
                                  median_recency_days=("recency_days", "median")).reset_index()
seg["pct_customers"] = seg["customers"] / seg["customers"].sum()
seg["pct_revenue"] = seg["revenue"] / seg["revenue"].sum()
seg["suggested_action"] = seg["segment"].map(actions)
seg = seg.sort_values("revenue", ascending=False)
seg.round(4).to_csv(f"{OUT}/rfm_segments.csv", index=False)


# 8. Other outputs for Power BI

out_c = cust.reset_index().rename(columns={"cust": "customer_id"})
out_c = out_c.drop(columns=["cohort_idx"])
out_c.to_csv(f"{OUT}/customers.csv", index=False)
sales[["invoice", "order_day", "date", "cust", "stock", "desc", "qty", "price", "rev", "country"]] \
    .rename(columns={"cust": "customer_id", "rev": "revenue"}).to_csv(f"{OUT}/transactions_clean.csv", index=False)


# 9. Summary file (small: paste or upload this one)

say("=" * 70)
say("DAY 1 SUMMARY: Who Comes Back?")
say("=" * 70)
say(f"Input file: {IN_PATH}")
say(f"Data range in file: {data_start.date()} to {data_end.date()}")
say(f"Analysis window: {data_start.date()} to {window_end.date()} (last full month)")
say(f"Exact duplicate rows in the raw file (reported, NOT removed): {n_dup}")
say("")
say("--- CLEANING LOG ---")
show(log_df)
say("")
say(f"Value of returns/cancellations set aside: {returns['rev'].sum():,.2f}")
say(f"Revenue with no Customer ID (excluded): {nocust['rev'].sum():,.2f} "
    f"({nocust['rev'].sum() / base_rev:.1%} of positive sales)")
say(f"Revenue analysed: {analysis_rev:,.2f}   |   customers analysed: {len(cust):,}")
top_countries = (sales.groupby("country")["rev"].sum().sort_values(ascending=False).head(3) / analysis_rev)
say("Top countries by revenue share: " + ", ".join(f"{k} {v:.1%}" for k, v in top_countries.items()))
srt = cust["revenue"].sort_values(ascending=False)
say(f"Revenue share of top 10% of customers: {srt.head(int(len(srt) * 0.1)).sum() / srt.sum():.1%}")

say("")
say("--- CHECKS ---")
checks = []
checks.append(("Month-0 retention is 100% for every cohort",
               bool(np.allclose(cr[cr["months_since_first"] == 0]["retention_rate"], 1.0))))
checks.append(("Cohort sizes add up to number of new customers", int(sizes["cohort_size"].sum()) == len(cn)))
checks.append(("No customer appears twice in customers table", cust.index.is_unique))
rec = base_rev - (nonprod["rev"].sum() + nocust["rev"].sum() + late["rev"].sum())
checks.append(("Revenue reconciles: positive sales = analysed + removed", abs(rec - analysis_rev) < 0.01))
checks.append(("All analysed dates fall inside the window", bool((sales["date"] <= window_end + pd.Timedelta(days=1)).all())))
for name, ok in checks:
    say(f"[{'PASS' if ok else 'FAIL'}] {name}")

say("")
say("--- HEADLINES ---")
say(f"Customers analysed (all): {len(cust):,}   of which first seen in {idx_label(first_cohort)} "
    f"(excluded from cohort/repeat/LTV, kept in RFM): {int((~cust['is_new']).sum()):,}")
say(f"New customers used for cohort/repeat/LTV: {len(cn):,}")
say(f"Orders (customer-days, all customers): {len(orders):,}   Avg order value: {orders['rev'].mean():,.2f}")
say(f"Share of NEW customers who ever bought a second time (not time-adjusted): {cn['repeat_ever'].mean():.1%}")
el = cust[cust["eligible_180d"]]  # new customers with 180+ days of history
say(f"Customers with at least {WIN_REPEAT} days of history: {len(el):,}")
say(f"  Second order within 30 days:  {el['days_to_second'].le(30).mean():.1%}")
say(f"  Second order within 90 days:  {el['days_to_second'].le(90).mean():.1%}")
say(f"  Second order within 180 days: {el['days_to_second'].le(180).mean():.1%}  (baseline used for gateway lift)")
rp = cn[cn["repeat_ever"]]["days_to_second"]
say(f"Median days to second order, among customers who did return: {rp.median():.0f}")
e365 = cust[cust["eligible_365d"]]
say(f"Customers with 365 days of history: {len(e365):,}")
say(f"  Average 365-day revenue per customer: {e365['ltv_365'].mean():,.2f}   median: {e365['ltv_365'].median():,.2f}")

say("")
say("--- RETENTION: share of cohort active in month N (weighted over cohorts where N is observable) ---")
for k in (1, 3, 6, 12):
    sub = cr[cr["months_since_first"] == k]
    if len(sub):
        say(f"Month {k}: {sub['customers_active'].sum() / sub['cohort_size'].sum():.1%}  ({len(sub)} cohorts)")

say("")
say("--- RETENTION BY COHORT (blank = not yet observable) ---")
piv = cr[cr["months_since_first"].isin([1, 3, 6, 12])].pivot(
    index="cohort_month", columns="months_since_first", values="retention_rate")
piv = piv.rename(columns=lambda k: f"m{k}")
piv.insert(0, "cohort_size", sizes.set_index(sizes["cohort_idx"].map(idx_label))["cohort_size"])
piv = (piv.reset_index())
for c in ("m1", "m3", "m6", "m12"):
    if c in piv:
        piv[c] = piv[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
show(piv)

say("")
say("--- FIRST-ORDER VALUE BANDS ---")
show(bands)

say("")
say(f"--- GATEWAY PRODUCTS (first order included this product; repeat within {WIN_REPEAT} days) ---")
say(f"Baseline repeat rate (all eligible customers): {baseline:.1%}.  Min customers per product: {MIN_N}")
ok = gw[gw["meets_min_customers"]]
say(f"Products meeting the minimum: {len(ok)}")
cols = ["stock", "description", "customers", "repeat_rate_180d", "lift_vs_baseline_pp"]
say("Top 15 by repeat rate:")
show(ok.head(15)[cols].round(3))
say("Bottom 8 by repeat rate:")
show(ok.tail(8)[cols].round(3))

say("")
say("--- RFM SEGMENTS ---")
show(seg.drop(columns=["suggested_action"]).round(3))

with open(f"{OUT}/summary.txt", "w", encoding="utf-8") as fh:
    fh.write("\n".join(lines))
print("\nDone. Outputs are in the folder:", OUT)


DAY 1 SUMMARY: Who Comes Back?
Input file: online_retail_II.csv
Data range in file: 2009-12-01 to 2011-12-09
Analysis window: 2009-12-01 to 2011-11-30 (last full month)
Exact duplicate rows in the raw file (reported, NOT removed): 34335

--- CLEANING LOG ---
                                                    step  rows_removed  value_removed  rows_remaining                                                                         note
                                             Rows loaded             0           0.00         1067371                                                                             
                     Unparseable date, quantity or price             0           0.00         1067371                                                                             
Cancellations/returns (invoice starts with C), set aside         19494    -1526667.86         1047877                    Value is negative. Kept separately, not netted off sales.
  Zero or negative quanti